# MultiScale CNEEP for SAOU Model

Estimates total exact entropy production (System + Medium EP) across spatial correlation scales.

In [ ]:
import sys
import os
CNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')
if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'SAOU'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from utils.sampler import CartesianSeqSampler
from tqdm import tqdm
import matplotlib.pyplot as plt
import joblib
from saou_model import simulate, pathwise_total_epr_increment, build_shell_ops, make_annular_shells, irreversible_velocity_total, theoretical_epr_gram


In [ ]:
# Hyper parameters
opt = Namespace()
opt.device = 'cuda' if torch.cuda.is_available() else 'cpu'
opt.alpha = -0.5
opt.lam = 0.0
opt.periodic = True
opt.positional = False
opt.n_components = 2
opt.n_iter = 2000
opt.train_batch_size = 4096
opt.test_batch_size = 2048
opt.video_batch_size = 256
opt.lr = 1e-4
opt.wd = 1e-5
opt.input_scalar = 1
opt.loss_scalar = 1
opt.scalar = 1
opt.clip_norm = 1
opt.max_distance = 10
opt.include_k0 = True
opt.beta = 1.0
opt.record_freq = 100
opt.seed = 3
opt.n_layer = 2
opt.n_channel = 8
opt.n_hidden = 2
opt.input_shape = (32, 32)
opt.M = 2
opt.M_test = 1
opt.L = 1000
opt.L_test = 10000
opt.seq_len = 2
opt.val_ratio = 0.2
opt.time_step = 0.001

# SAOU Parameters
saou_kwargs = dict(
    L=32,
    radii=tuple(range(1, opt.max_distance + 1)),
    amplitudes=(1.0, 0.8, 1.2, 0.5, -0.5, 0.0, 0.5, -0.2, 0.3, 0.1),
    gamma=1.0,
    omega0=1.0,
    T=1.0,
    dt=0.001,
    sample_every=10,
)
n_steps = opt.L * saou_kwargs['sample_every']
burn_steps = 10000
dt_eff = saou_kwargs['dt'] * saou_kwargs['sample_every']

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(result_folder, f'CorrSAOU-{datetime.now().strftime("%Y-%m-%d-%H%M%S")}')
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')
best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')
print(f'Device: {opt.device}')
print(f'Results: {current_result_folder}')


In [ ]:
print(f'[INFO] Generating TRAIN trajectories (M={opt.M}) on {opt.device}...')
res_train = simulate(
    **saou_kwargs,
    n_steps=n_steps,
    burn_steps=burn_steps,
    seed=opt.seed,
    M=opt.M,
    device=opt.device,
    record_trajectory=True,
    show_progress=True
)
traj_train = res_train['trajectory']
if traj_train.ndim == 4: traj_train = traj_train[:, np.newaxis, ...]
trajectories_train = np.transpose(traj_train, (1, 0, 2, 3, 4)) # (M, L, Lx, Ly, 2)
print('Train shape:', trajectories_train.shape)

print(f'[INFO] Generating TEST trajectory (M={opt.M_test}) on {opt.device}...')
test_steps = opt.L_test * saou_kwargs['sample_every']
res_test = simulate(
    **saou_kwargs,
    n_steps=test_steps,
    burn_steps=burn_steps,
    seed=opt.seed + opt.M,
    M=opt.M_test,
    device=opt.device,
    record_trajectory=True,
    show_progress=True
)
traj_test = res_test['trajectory']
if traj_test.ndim == 4: traj_test = traj_test[:, np.newaxis, ...]
trajectories_test = np.transpose(traj_test, (1, 0, 2, 3, 4)) # (M_test, L_test, Lx, Ly, 2)
print('Test shape:', trajectories_test.shape)


In [ ]:
# Reorder channels to (M, L, 2, Lx, Ly)
if trajectories_train.ndim == 5 and trajectories_train.shape[-1] == 2:
    trajectories_train = np.transpose(trajectories_train, (0, 1, 4, 2, 3))
if trajectories_test.ndim == 5 and trajectories_test.shape[-1] == 2:
    trajectories_test = np.transpose(trajectories_test, (0, 1, 4, 2, 3))

train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

train_video = torch.from_numpy(trajectories_train[:M_train_new]).float().to(opt.device)
val_video = torch.from_numpy(trajectories_train[M_train_new:]).float().to(opt.device)
test_video = torch.from_numpy(trajectories_test).float().to(opt.device)

print('Train video:', train_video.shape)
print('Val video:', val_video.shape)
print('Test video:', test_video.shape)


In [ ]:
from models.NEEP_Corr_2D import MultiScaleCNEEP2D
from livelossplot import PlotLosses

mean = torch.mean(train_video, dim=(0, 1, 3, 4), keepdim=True)
std = torch.std(train_video, dim=(0, 1, 3, 4), keepdim=True)
transform = lambda x: (x - mean.to(x.device)) * opt.input_scalar / std.to(x.device)

model = MultiScaleCNEEP2D(opt).to(opt.device)
optim = torch.optim.Adam(model.parameters(), opt.lr, weight_decay=opt.wd)

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

train_sampler = CartesianSeqSampler(M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
val_sampler = CartesianSeqSampler(M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False)

best_val_loss = float('inf')
best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')

liveloss = PlotLosses()
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None

for it in tqdm(range(1, opt.n_iter + 1)):
    model.train()
    batch = next(train_sampler)
    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.stack(slices, dim=1).float().to(opt.device))

    J_all = model(x) / opt.scalar
    ent_production = J_all.sum(dim=1)

    optim.zero_grad()
    if opt.alpha == 0:
        loss = (- ent_production + (torch.exp(-ent_production) - 1)).mean()
    else:
        loss = (- (torch.exp(opt.alpha * ent_production) - 1) / opt.alpha
                + (torch.exp(-(1 + opt.alpha) * ent_production) - 1) / (1 + opt.alpha)).mean()

    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()

    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.stack(vslices, dim=1).float().to(opt.device))
                vJ = model(vx) / opt.scalar
                v_ep = vJ.sum(dim=1)
                if opt.alpha == 0:
                    vloss = (- v_ep + (torch.exp(-v_ep) - 1)).sum().item()
                else:
                    vloss = (- (torch.exp(opt.alpha * v_ep) - 1) / opt.alpha
                            + (torch.exp(-(1 + opt.alpha) * v_ep) - 1) / (1 + opt.alpha)).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]

        avg_val = val_loss_acc / n_val
        
        # Save checkpoint
        state = {
            'settings': opt.__dict__,
            'state_dict': model.state_dict(),
            'optimizer': optim.state_dict(),
            'iteration': it,
        }
        torch.save(state, current_checkpoint_path)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, best_checkpoint_path)

        if smooth_train_loss is None:
            smooth_train_loss = loss.item()
            smooth_val_loss = avg_val
        else:
            smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
            smooth_val_loss = smoothing * smooth_val_loss + (1 - smoothing) * avg_val
        liveloss.update({'train_loss': smooth_train_loss, 'val_loss': smooth_val_loss})
        liveloss.send()
print('Training finished.')
print(f'Checkpoint saved: {current_checkpoint_path}')


In [ ]:
# Load the Best or Final model
load_best = True  # Set to False to load the model from the final iteration

best_checkpoint_path = os.path.join(current_result_folder, 'best_model_parameter.pth.tar')
if load_best and os.path.exists(best_checkpoint_path):
    print("Loading BEST model (lowest validation loss)...")
    checkpoint = torch.load(best_checkpoint_path, map_location=opt.device)
else:
    print("Loading FINAL iteration model...")
    checkpoint = torch.load(current_checkpoint_path, map_location=opt.device)

model.load_state_dict(checkpoint['state_dict'])
model.eval()
print(f"Loaded model from iteration {checkpoint.get('iteration', 'Unknown')}")

In [ ]:
# Ground Truth Computation
# Calculate System EP + Medium EP explicitly without window summation
shells = make_annular_shells(saou_kwargs['radii'], saou_kwargs['amplitudes'])
ops = build_shell_ops(saou_kwargs['L'], shells)
gamma = saou_kwargs['gamma']
T_temp = saou_kwargs['T']
omega0 = saou_kwargs['omega0']
L_size = saou_kwargs['L']

# traj_test_raw shape: (M_test, L_test, L_size, L_size, 2)
traj_test_raw = trajectories_test.transpose(0, 1, 3, 4, 2)

n_windows = opt.L_test - 1
gt_total_epr = np.zeros(n_windows)
gt_epr_map_sum = np.zeros((L_size, L_size))

print('[INFO] Computing GT Exact EP Map for TEST data...')
for t in tqdm(range(n_windows)):
    x0 = traj_test_raw[0, t]     # (Lx, Ly, 2)
    x1 = traj_test_raw[0, t+1]
    x_mid = 0.5 * (x0 + x1)
    dx = x1 - x0
    
    # System Entropy increment: (gamma / 2T) * (|x1|^2 - |x0|^2)
    ds_sys = (gamma / (2.0 * T_temp)) * (np.sum(x1**2, axis=-1) - np.sum(x0**2, axis=-1))
    
    # Medium Entropy increment: (1 / T) * (v_irr(x_mid) dot dx)
    x_mid_t = torch.from_numpy(x_mid).to(opt.device)
    vel_irr = irreversible_velocity_total(x_mid_t, ops, omega0).cpu().numpy()
    ds_med = (1.0 / T_temp) * np.sum(vel_irr * dx, axis=-1)
    
    # Total local EP
    ds_total = ds_sys + ds_med
    
    gt_epr_map_sum += ds_total
    gt_total_epr[t] = np.sum(ds_total)

gt_epr_maps = (gt_epr_map_sum / n_windows)[np.newaxis, ...] # (1, Lx, Ly)
print(f'GT mean EPR rate: {gt_total_epr.mean() / dt_eff:.6e}')


In [ ]:
model.eval()
pred_total_ep = []
test_sampler = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False)

with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.stack(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar
        pred_total_ep.append(J_all.sum(dim=1).cpu().numpy())

pred_total_ep = np.concatenate(pred_total_ep)
min_len = min(len(gt_total_epr), len(pred_total_ep))
time_axis = np.arange(min_len) * dt_eff

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
axes[0].plot(time_axis, gt_total_epr[:min_len] / dt_eff, lw=0.5, alpha=0.6, label='GT EPR')
axes[0].plot(time_axis, pred_total_ep[:min_len] / dt_eff, lw=0.5, alpha=0.6, label='Pred EPR')
axes[0].set_ylabel('EPR')
axes[0].legend()
axes[0].set_title('Instantaneous EPR Comparison')

axes[1].plot(time_axis, np.cumsum(gt_total_epr[:min_len]), label='GT Cumul EP')
axes[1].plot(time_axis, np.cumsum(pred_total_ep[:min_len]), label='Pred Cumul EP')
axes[1].set_ylabel('Cumulative EP')
axes[1].legend()

window = 100
gt_smooth = np.convolve(gt_total_epr[:min_len] / dt_eff, np.ones(window)/window, mode='same')
pred_smooth = np.convolve(pred_total_ep[:min_len] / dt_eff, np.ones(window)/window, mode='same')
axes[2].plot(time_axis, gt_smooth, label='GT (Smooth)')
axes[2].plot(time_axis, pred_smooth, label='Pred (Smooth)')
axes[2].set_ylabel('Running Avg')
axes[2].set_xlabel('Time')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{current_result_folder}/epr_timeseries.png', dpi=150)
plt.show()


In [ ]:
model.eval()
test_sampler_one = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, 1, device=opt.device, train=False)
ens_idx, traj_idx = next(test_sampler_one)
b0 = ens_idx.to(test_video.device)
slices = [test_video[(b0, traj_idx[i].to(test_video.device))] for i in range(opt.seq_len)]
x = transform(torch.stack(slices, dim=1).float().to(opt.device))

with torch.no_grad():
    maps = model(x, return_maps=True) / opt.scalar # [B, K+1, Lx, Ly]

pred_map_k = maps[0].cpu().numpy() / dt_eff # [K+1, Lx, Ly]
pred_total_map = pred_map_k.sum(axis=0)
gt_map = gt_epr_maps[0] / dt_eff

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vmax = max(np.abs(gt_map).max(), np.abs(pred_total_map).max(), 1e-12)
im1 = axes[0].imshow(gt_map.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('GT EP Map (Time Avg)')
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(pred_total_map.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Predicted Total EP Map (Single Step)')
fig.colorbar(im2, ax=axes[1])
plt.tight_layout()
plt.savefig(f'{current_result_folder}/local_ep_map_2d.png', dpi=150)
plt.show()


In [ ]:
print('[INFO] Calculating Ensemble Averaged Map...')
all_maps = []
test_sampler_batch = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False)

with torch.no_grad():
    for batch in tqdm(test_sampler_batch):
        b0 = batch[0].to(test_video.device)
        vslices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        vx = transform(torch.stack(vslices, dim=1).float().to(opt.device))
        m = model(vx, return_maps=True) / opt.scalar
        all_maps.append(m.cpu().numpy())

all_maps = np.concatenate(all_maps, axis=0) / dt_eff # [N, K+1, Lx, Ly]
ensemble_map_k = all_maps.mean(axis=0)
ensemble_pred_total = ensemble_map_k.sum(axis=0)
ensemble_gt = gt_epr_maps[0] / dt_eff

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
vmax = max(np.abs(ensemble_gt).max(), np.abs(ensemble_pred_total).max(), 1e-12)
im0 = axes[0].imshow(ensemble_gt.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('GT Ensemble Mean EP Map')
fig.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(ensemble_pred_total.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Predicted Ensemble Total EP Map')
fig.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map_2d.png', dpi=150)
plt.show()

cols = min(opt.max_distance + 1, 6)
rows = int(np.ceil((opt.max_distance + 1) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
for k in range(opt.max_distance + 1):
    im = axes[k].imshow(ensemble_map_k[k].T, origin='lower', cmap='RdBu_r')
    axes[k].set_title(f'k={k}')
    fig.colorbar(im, ax=axes[k], shrink=0.6)
for i in range(opt.max_distance + 1, len(axes)):
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ensemble_ep_map_2d_k.png', dpi=150)
plt.show()


In [ ]:
model.eval()
all_J = []
test_sampler = CartesianSeqSampler(opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False)
with torch.no_grad():
    for batch in test_sampler:
        b0 = batch[0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.stack(slices, dim=1).float().to(opt.device))
        J = model(x) / opt.scalar
        all_J.append(J.cpu().numpy())

all_J = np.concatenate(all_J, axis=0)
mean_J = all_J.mean(axis=0)
std_J  = all_J.std(axis=0)
distances = np.arange(0, opt.max_distance + 1)

gt_gram = theoretical_epr_gram(saou_kwargs['L'], shells, saou_kwargs['gamma'], saou_kwargs['omega0'])
gt_components = gt_gram.sum(axis=1)
gt_labels = ['Local'] + [s.name for s in shells]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(distances, mean_J / dt_eff, yerr=std_J/dt_eff/np.sqrt(len(all_J)), capsize=3, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Kernel distance $k$')
axes[0].set_ylabel('$\\langle J_k \\rangle / dt_{eff}$')
axes[0].set_title('Predicted EP Spectrum (by distance)')
axes[0].set_xticks(distances)

axes[1].bar(range(len(gt_components)), gt_components, color='mediumseagreen', alpha=0.7)
axes[1].set_xticks(range(len(gt_components)))
axes[1].set_xticklabels(gt_labels, rotation=45, ha='right')
axes[1].set_ylabel('Theoretical EPR rate')
axes[1].set_title('Ground Truth EP Spectrum (by shell)')

cum_J = np.cumsum(mean_J / dt_eff)
axes[2].plot(distances, cum_J, 'o-', color='darkorange', label='Predicted Cumulative')
axes[2].axhline(y=gt_components.sum(), color='red', linestyle='--', label='GT Total EPR')
axes[2].set_xlabel('Kernel distance $k$')
axes[2].set_ylabel('Cumulative EPR')
axes[2].set_title('Cumulative EPR')
axes[2].set_xticks(distances)
axes[2].legend()
plt.tight_layout()
plt.savefig(f'{current_result_folder}/ep_spectrum_2d.png', dpi=150)
plt.show()

print(f'Total predicted EPR: {mean_J.sum() / dt_eff:.6e}')
for k in range(opt.max_distance + 1):
    print(f'  k={k}: J_k / dt_eff = {mean_J[k] / dt_eff:.6e} ({100 * mean_J[k] / mean_J.sum():.1f}%)')
print(f'\nTotal GT EPR: {gt_components.sum():.6e}')
